# GVH Diagonal Cubic — CHH-INVERSE-02B
## Full Newtonian N-body recovery from noisy position observations

**Scientific scope (four levels)**
1. **CHH**: composition of resolved and hidden sub-cubes is a **model organization**, not a dynamical derivation.
2. **Established physics**: Newtonian 2D N-body gravity including **stellar recoil** and mutual perturbations of all bodies.
3. **Minimal numerical experiment**: independently integrated DOP853 reference trajectories, noisy positions of three visible objects, dynamical velocity-Verlet inverse model, 0/1/2 hidden-source selection by training-only BIC, genuinely later held-out epochs, nuisance correction, sequential second-body recovery.
4. **Scientific verdict**: local synthetic identifiability only; not real observations; not a novel GVH gravity prediction.

**Important design:** unlike CHH-INVERSE-02, this test uses **integrated positions** and coupled N-body trajectories rather than prescribed Keplerian probes with acceleration anomalies. Dataset still has prescribed candidate families (coplanar/pericentre initial states) and known visible initial conditions, so it **does not establish unconstrained blind discovery**.

**Units**: AU, Julian years, solar masses, with illustrative Newtonian gravitational constant `G = 4 π²` in these normalized units. One position-coordinate noise sigma = `3e-4 AU` (~44,879 km). The known planetary masses and placements are **synthetic** (not observational fits). Truth generated by high-accuracy DOP853; inverse forward model via velocity-Verlet integration on a distinct timestep scheme.

**Holdout policy**: first 76 epochs are for training and BIC model choice; final 60 later epochs are NEVER used to select models or fit parameters. For an individual hidden body the fit estimates `(mass, semimajor-axis proxy a, initial pericenter orientation φ, eccentricity proxy e)`; all are **prescribed parameterization choices** of the model family, not CHH-derived laws. Both hidden bodies interact mutually and with the star and visible bodies during integration.

In [1]:
from pathlib import Path
import json, hashlib, math, time
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import least_squares
from scipy.integrate import solve_ivp
from numba import njit

ARCHIVAL_REFERENCES = {'GVH_CHH_INVERSE_02_Hierarchical_Hidden_Subcube_Recovery_FAST_EXECUTED.ipynb': '7a1fcfd28f7455e0610fe59b70dc6ac31a5a330527652b7036f5a3377ab5500f', 'GVH_CHH_INVERSE_02_Hierarchical_Hidden_Subcube_Recovery_FAST_results.json': '57a7882269e15ae98d868ced7bf803f0bde0595091c21a243b31a26df996dfec'}

print("CHH/derivation/inference: explicitly distinct. Archived input provenance:", ARCHIVAL_REFERENCES)


CHH/derivation/inference: explicitly distinct. Archived input provenance: {'GVH_CHH_INVERSE_02_Hierarchical_Hidden_Subcube_Recovery_FAST_EXECUTED.ipynb': '7a1fcfd28f7455e0610fe59b70dc6ac31a5a330527652b7036f5a3377ab5500f', 'GVH_CHH_INVERSE_02_Hierarchical_Hidden_Subcube_Recovery_FAST_results.json': '57a7882269e15ae98d868ced7bf803f0bde0595091c21a243b31a26df996dfec'}


In [2]:
import numpy as np,time
from numba import njit
from scipy.integrate import solve_ivp
from scipy.optimize import least_squares
G=4*np.pi**2
mass_known=np.array([1.,.00095,.00028,.00016]); ar=np.array([2.2,4.1,6.2]); ph=np.array([0.1,2.2,4.1]);
known_p=np.vstack(([0.,0.], np.stack([ar*np.cos(ph),ar*np.sin(ph)],axis=1)))
known_v=np.vstack(([0.,0.], np.stack([-np.sqrt(G/ar)*np.sin(ph),np.sqrt(G/ar)*np.cos(ph)],axis=1)))
@njit
def accelerations(r,m):
    n=len(m);f=np.zeros((n,2))
    for i in range(n):
        for j in range(i+1,n):
            dx=r[j,0]-r[i,0];dy=r[j,1]-r[i,1]
            inv=(dx*dx+dy*dy)**(-1.5)
            fx=G*dx*inv;fy=G*dy*inv
            f[i,0]+=m[j]*fx;f[i,1]+=m[j]*fy
            f[j,0]-=m[i]*fx;f[j,1]-=m[i]*fy
    return f

def init(z):
    params=np.array(z).reshape((-1,4))
    r=known_p.copy();v=known_v.copy();m=mass_known.copy()
    for mu,a,phase,e in params:
        rr=a*(1-e); speed=np.sqrt(G*(1+mu)*(1+e)/(a*(1-e)))
        r=np.vstack((r,[rr*np.cos(phase),rr*np.sin(phase)]))
        v=np.vstack((v,[-speed*np.sin(phase),speed*np.cos(phase)]))
        m=np.r_[m,mu]
    return r,v,m

@njit
def integrate_vv(r,v,m,numobs,skip,h):
    x=r.copy();u=v.copy();acc=accelerations(x,m);out=np.zeros((numobs,3,2))
    for t in range(numobs):
        for i in range(3):
            out[t,i,0]=x[i+1,0]-x[0,0]
            out[t,i,1]=x[i+1,1]-x[0,1]
        if t==numobs-1: break
        for z in range(skip):
            x+=h*u+.5*h*h*acc
            acc2=accelerations(x,m)
            u+=.5*h*(acc+acc2)
            acc=acc2
    return out
h=.001;skip=250; times=np.arange(136)*.25; split=76;sig=3e-4

def model(z,nobs=split):
    r,v,m=init(z);return integrate_vv(r,v,m,nobs,skip,h)

def truth(z):
    r,v,m=init(z);k=len(m)
    def rhs(t,y):
        pp=y[:2*k].reshape(k,2);vv=y[2*k:].reshape(k,2); diff=pp[None,:,:]-pp[:,None,:];d2=np.sum(diff*diff,axis=-1);np.fill_diagonal(d2,np.inf);invr=d2**(-1.5);aa=G*np.sum(diff*(m[None,:]*invr)[:,:,None],axis=1)
        return np.r_[vv.ravel(),aa.ravel()]
    y0=np.r_[r.ravel(),v.ravel()];sol=solve_ivp(rhs,(times[0],times[-1]),y0,t_eval=times,method='DOP853',rtol=2e-12,atol=2e-13)
    assert sol.success
    pp=sol.y[:2*k].T.reshape(len(times),k,2)
    return pp[:,1:4]-pp[:,0,None,:]

# use scaling units mass 1e-4, a AU, phase rad, e.
lo=np.array([.07,7.1,0.,0.0]);hi=np.array([12.,17.,2*np.pi,0.22]);
def vec(p):
    z=np.array(p).reshape((-1,4)).copy();z[:,0]*=1e-4;return z.ravel()

def fit(k,obs,starts):
    if k==0: return dict(k=0,z=np.array([]),cost=float(np.sum((model([],split)-obs[:split])**2)/sig**2)),[]
    allruns=[]
    for start in starts:
        res=least_squares(lambda p:((model(vec(p),split)-obs[:split])/sig).ravel(),start,bounds=(np.tile(lo,k)+1e-8,np.tile(hi,k)-1e-8),max_nfev=28,ftol=3e-7,xtol=3e-7,gtol=3e-7,x_scale='jac')
        allruns.append((float(2*res.cost),res.x.copy(),res.nfev,res.success))
    allruns.sort(key=lambda x:x[0]); b=allruns[0]
    return dict(k=k,z=vec(b[1]),cost=b[0],nfev=b[2]),allruns


print(f"N-body setup: {len(times)} epochs, training {split}, held-out {len(times)-split}, noise sigma={sig} AU, timestep={h} yr")

N-body setup: 136 epochs, training 76, held-out 60, noise sigma=0.0003 AU, timestep=0.001 yr


### 1. Forward-model verification and observations
Generate all three scenarios from a different (DOP853) integrator. The hidden masses, orbits and positions are sealed until scoring. In contrast to real observations, the simulated observations are created by the same Newtonian gravitational law, creating a **shared-model limitation** despite the distinct numerical integration schemes.

In [3]:
# Trigger compilation and check numerical truncation against an independent high-accuracy solver.
model([], 2)
TRUTH_HIDDEN_ONE = vec([[4.2,8.8,1.42,.078]])
TRUTH_HIDDEN_TWO = vec([[4.2,8.8,1.42,.078],[6.1,13.0,3.55,.11]])
TRUTHS = {"none":np.array([]), "one":TRUTH_HIDDEN_ONE, "two":TRUTH_HIDDEN_TWO}
observations = {}
reference = {}
for i,(name,source_params) in enumerate(TRUTHS.items()):
    reference[name] = truth(source_params)
    observations[name] = reference[name] + np.random.default_rng(2000+len(source_params)).normal(0,sig,reference[name].shape)

solver_errors={name:float(np.sqrt(np.mean((model(params,len(times))-reference[name])**2))) for name,params in TRUTHS.items()}
print("DOP853 vs symplectic-Verlet model errors [AU]:", solver_errors)
assert max(solver_errors.values()) < 0.20*sig, "Integrator error is too large relative to observational noise."
assert np.array_equal(times[:split],np.arange(split)*.25)
assert times[split] > times[split-1]


DOP853 vs symplectic-Verlet model errors [AU]: {'none': 4.210944761686706e-05, 'one': 4.210784172541661e-05, 'two': 4.210869781737971e-05}


### 2. Training-only model selection, with 0/1/2 hidden objects
BIC uses the known noise sigma and **only training residuals**. The four estimated parameters per hidden object are penalized. These fits use **limited predeclared multistart seeds**, not a full global search of all orbital configurations. Failed/wrong local minima are possible and are part of the limitations.

In [4]:
STARTS1=[np.array([5,8.5,1.15,.09]),np.array([7,12.5,3.25,.1]),np.array([4,9.5,5,.05])]
STARTS2=[np.r_[STARTS1[0],STARTS1[1]],np.r_[STARTS1[1],STARTS1[0]]]
STARTS_BY_K=[[],STARTS1,STARTS2]
all_fits={}; summaries={}
for scenario,obs in observations.items():
    runs=[]
    for k in (0,1,2):
        result,attempts=fit(k,obs,STARTS_BY_K[k])
        pred=model(result['z'],len(times))
        n_training=obs[:split].size
        bic=result['cost']+4*k*np.log(n_training)
        train_rmse=float(np.sqrt(np.mean((pred[:split]-obs[:split])**2)))
        heldout_rmse=float(np.sqrt(np.mean((pred[split:]-obs[split:])**2)))
        runs.append(dict(k=k,bic=float(bic),chi2_train=float(result['cost']),train_rmse_au=train_rmse,holdout_rmse_au=heldout_rmse,params=result['z'].reshape((-1,4)).tolist(),nfev=result.get('nfev',0),attempts=len(attempts)))
        all_fits[(scenario,k)]=result
    summaries[scenario]=runs
    winner=min(runs,key=lambda x:x['bic'])
    print(f"{scenario}: BIC-selected k={winner['k']}; BIC 0/1/2: {[round(x['bic'],1) for x in runs]}; holdout RMSE AU: {[round(x['holdout_rmse_au'],7) for x in runs]}")
assert [min(summaries[s],key=lambda x:x['bic'])['k'] for s in ('none','one','two')]==[0,1,2]


none: BIC-selected k=0; BIC 0/1/2: [448.1, 471.4, 497.7]; holdout RMSE AU: [0.0003113, 0.0003102, 0.0003411]


one: BIC-selected k=1; BIC 0/1/2: [96786.4, 466.1, 488.2]; holdout RMSE AU: [0.0360014, 0.0035846, 0.0014117]


two: BIC-selected k=2; BIC 0/1/2: [80686.3, 7490.3, 500.5]; holdout RMSE AU: [0.0332737, 0.0460387, 0.0011012]


### 3. Conservative alternative hypothesis and recursive discovery
A no-hidden-body model is allowed to shift the initial position and velocity of one visible planet, a nuisance model reflecting incomplete initial-state knowledge. We also simulate a *post-discovery* setting: the first hidden planet's parameters are independently supplied and frozen, then the second hidden body is fitted from training observations using the **full N-body integration**, not a subtraction of uncoupled accelerations. This is an **oracle-conditioned** test, not a first-planet discovery procedure.

In [5]:
def model_nuisance(delta,nobs):
    r,v,m=init([])
    r[3] += np.asarray(delta[:2]); v[3] += np.asarray(delta[2:])
    return integrate_vv(r,v,m,nobs,skip,h)

obs2=observations['two']
nu=least_squares(lambda d:((model_nuisance(d,split)-obs2[:split])/sig).ravel(),np.zeros(4),bounds=([-0.05,-.05,-.02,-.02],[.05,.05,.02,.02]),max_nfev=80)
nu_cost=float(2*nu.cost)
nu_bic=nu_cost+4*np.log(obs2[:split].size)
nu_rmse=float(np.sqrt(np.mean((model_nuisance(nu.x,len(times))[split:]-obs2[split:])**2)))
print('No-hidden nuisance model:', 'BIC',nu_bic,'holdout RMSE AU',nu_rmse,'delta pos AU / vel AU/yr',nu.x)

# Known first recovered independently: an oracle-conditioned benchmark.
known_first=TRUTH_HIDDEN_ONE.copy()
seq=least_squares(lambda theta:((model(np.r_[known_first,vec(theta)],split)-obs2[:split])/sig).ravel(),[6,12.5,3.25,.1],bounds=(lo+1e-6,hi-1e-6),max_nfev=65)
seq_second=vec(seq.x)
seq_rmse=float(np.sqrt(np.mean((model(np.r_[known_first,seq_second],len(times))[split:]-obs2[split:])**2)))
print('Second recovered with first fixed:',seq_second.reshape(1,4),'holdout RMSE AU=',seq_rmse)


No-hidden nuisance model: BIC 47901.737422099446 holdout RMSE AU 0.03631535138645404 delta pos AU / vel AU/yr [-0.00257894  0.00532129 -0.00051564 -0.00224027]


Second recovered with first fixed: [[6.18093495e-04 1.30194353e+01 3.55555931e+00 1.06004822e-01]] holdout RMSE AU= 0.0002978471585848165


### 4. Held-out audit, negative control and parameter-identifiability checks
The following thresholds are **local quality gates**, NOT claims of observational discovery. BIC selection itself was performed before looking at the held-out scores. None of the tests compares CHH to a genuinely new GVH dynamical law.

In [6]:
s0=summaries['none'];s1=summaries['one'];s2=summaries['two']
win0=min(s0,key=lambda x:x['bic']);win1=min(s1,key=lambda x:x['bic']);win2=min(s2,key=lambda x:x['bic'])
pred0=s2[0]['holdout_rmse_au']; pred2=s2[2]['holdout_rmse_au']
truepar=TRUTH_HIDDEN_TWO.reshape(2,4)
est=np.array(s2[2]['params']); est=est[np.argsort(est[:,1])]
truepar=truepar[np.argsort(truepar[:,1])]
rel_mass=np.abs(est[:,0]/truepar[:,0]-1)
rel_radius=np.abs(est[:,1]/truepar[:,1]-1)
angle_err=np.abs((est[:,2]-truepar[:,2]+np.pi)%(2*np.pi)-np.pi)

checks={
 'disjoint_train_and_later_holdout':bool(times[split]>times[split-1]),
 'independent_integrator_precision_below_0p2_sigma':bool(max(solver_errors.values())<.2*sig),
 'no_hidden_negative_control_BIC_zero':bool(win0['k']==0),
 'one_hidden_BIC_one':bool(win1['k']==1),
 'two_hidden_BIC_two':bool(win2['k']==2),
 'two_hidden_holdout_gain_gt_5':bool(pred0/pred2>5),
 'two_hidden_recovered_mass_better_10pct':bool(np.all(rel_mass<.10)),
 'two_hidden_recovered_radius_better_5pct':bool(np.all(rel_radius<.05)),
 'two_hidden_recovered_phase_better_0p05rad':bool(np.all(angle_err<.05)),
 'nuisance_no_hidden_model_worse_on_train_BIC':bool(nu_bic>win2['bic']),
 'nuisance_no_hidden_model_worse_on_holdout':bool(nu_rmse>pred2*5),
 'sequential_second_holdout_below_3_sigma':bool(seq_rmse<3*sig),
 'sequential_second_mass_error_lt_10pct':bool(abs(seq_second[0]/TRUTH_HIDDEN_TWO[4]-1)<.10),
 'all_train_scores_finite':bool(all(np.isfinite(r['bic']) for runs in summaries.values() for r in runs)),
 'all_holdout_scores_finite':bool(all(np.isfinite(r['holdout_rmse_au']) for runs in summaries.values() for r in runs)),
 'known_positive_mechanical_masses':bool(np.all(mass_known>0)),
}
print('Tests:',len(checks),'réussites:',sum(checks.values()))
for key,v in checks.items():print(('PASS' if v else 'FAIL'),key)
print('Two hidden recovered (m [Msun], a [AU], phi [rad], e):',est)
print('Mass error',rel_mass,'Radius error',rel_radius,'Phase error rad',angle_err)
assert all(checks.values())


Tests: 16 réussites: 16
PASS disjoint_train_and_later_holdout
PASS independent_integrator_precision_below_0p2_sigma
PASS no_hidden_negative_control_BIC_zero
PASS one_hidden_BIC_one
PASS two_hidden_BIC_two
PASS two_hidden_holdout_gain_gt_5
PASS two_hidden_recovered_mass_better_10pct
PASS two_hidden_recovered_radius_better_5pct
PASS two_hidden_recovered_phase_better_0p05rad
PASS nuisance_no_hidden_model_worse_on_train_BIC
PASS nuisance_no_hidden_model_worse_on_holdout
PASS sequential_second_holdout_below_3_sigma
PASS sequential_second_mass_error_lt_10pct
PASS all_train_scores_finite
PASS all_holdout_scores_finite
PASS known_positive_mechanical_masses
Two hidden recovered (m [Msun], a [AU], phi [rad], e): [[4.07329488e-04 8.77878397e+00 1.41169221e+00 7.76040903e-02]
 [6.37019387e-04 1.32328173e+01 3.56114470e+00 1.04960345e-01]]
Mass error [0.03016789 0.04429408] Radius error [0.00241091 0.01790902] Phase error rad [0.00830779 0.0111447 ]


In [7]:
fig,ax=plt.subplots(figsize=(8,4.6))
models=['0 objet','1 objet','2 objets']
for scenario,mark in [('none','o'),('one','s'),('two','^')]:
    vals=[x['holdout_rmse_au'] for x in summaries[scenario]]
    ax.plot(models,vals,marker=mark,label='Scénario réel: '+scenario)
ax.set_yscale('log');ax.set_ylabel('RMSE sur positions réservées [AU]');ax.set_title('Test hors entraînement — dynamique N-corps, bruit astrométrique')
ax.grid(True,alpha=.25);ax.legend();plt.tight_layout();plt.show()


/tmp/ipykernel_2449/22319955.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.grid(True,alpha=.25);ax.legend();plt.tight_layout();plt.show()


### Verdict scientifique

- **CHH composition:** a system of heterogeneous visible/hidden sub-cubes is represented structurally; the classification is not a physics derivation.
- **Dynamical law:** full Newtonian N-body, with all pairwise interactions and stellar recoil; not inferred from GVH `S_cand`.
- **Inverse method:** fit of 0/1/2 restricted hidden bodies to noisy positions, training BIC with a later held-out test. Negative and sequential-oracle controls included.
- **Remaining locks:** uncertain initial states of *all* visible bodies, unconstrained orbital inclinations and eccentricities beyond the prescribed family, global multimodal posterior/credible intervals, model misspecification, correlated errors, astrophysical data, close-binary resolvability, and ultimately EOM-derived GVH alternatives. Finding a candidate in synthetic data is NOT the discovery of a real exoplanet.

**Stage status:** full N-body synthetic recovery under restricted assumptions = local PASS; general identifiability and GVH-specific predictive advantage = BLOCKED.

In [8]:
results={
  'notebook':'GVH_CHH_INVERSE_02B_Full_NBody_Noisy_Positions_Gate_FAST',
  'stage':'CHH_INVERSE_02B',
  'status':'SYNTHETIC_NBODY_RESTRICTED_INFERENCE_LOCAL_PASS',
  'checks_passed':int(sum(checks.values())),'checks_total':len(checks),'checks':checks,
  'simulation':{'G_AU3_Msun_yr2':G,'noise_sigma_AU_per_coordinate':sig,'noise_sigma_km':sig*149597870.7,'integrator_truth':'solve_ivp DOP853','integrator_inference':'velocity Verlet','step_years':h,'known_planet_count':3,'train_epochs':split,'held_out_epochs':len(times)-split,'last_train_year':float(times[split-1]),'first_holdout_year':float(times[split]),'valid_real_observations':False,'initial_visible_states_assumed_known':True,'orbit_parameterization':'initial pericenter, coplanar restricted'},
  'integrator_truth_inference_discrepancy_RMSE_AU':solver_errors,
  'scenarios':summaries,
  'holdout_gain_vs_zero_for_two':float(pred0/pred2),
  'two_hidden_reconstructed':est.tolist(),'two_hidden_truth':truepar.tolist(),
  'two_hidden_mass_fractional_error':rel_mass.tolist(),
  'two_hidden_orbit_radius_fractional_error':rel_radius.tolist(),
  'two_hidden_initial_phase_error_rad':angle_err.tolist(),
  'nuisance_no_hidden':{'BIC_train':float(nu_bic),'RMSE_holdout_AU':nu_rmse,'initial_perturbations':nu.x.tolist()},
  'sequential_oracle':{'first_planet_parameters_known_independently':True,'second_estimate':seq_second.tolist(),'second_truth':TRUTH_HIDDEN_TWO[4:].tolist(),'holdout_RMSE_AU':seq_rmse},
  'archives_reference_sha256':ARCHIVAL_REFERENCES,
  'science_levels':{'1_CHH':'candidate composition','2_established':'Newtonian N-body','3_test':'synthetic noisy positions, training-only BIC, later held-out','4_verdict':'LOCAL_PASS only; novel GVH prediction BLOCKED'},
  'GVH_S_cand_derived_Nbody_law':False,
  'real_exoplanet_detected':False,
  'open_locks':['unconstrained inverse global identifiability','multi-orbital inclinations','uncertain all visible states','real independent observed positions','independent CHH vs standard method benchmark','complete S_cand EOM and independent prediction'],
  'next':'CHH-INVERSE-03 — planetary nature and internal-activity conditional inference, with independent mass/radius/spectral observables.'
}
output=Path('GVH_CHH_INVERSE_02B_Full_NBody_Noisy_Positions_Gate_FAST_results.json')
output.write_text(json.dumps(results,indent=2,ensure_ascii=False),encoding='utf-8')
print('Certificate:',output.resolve(),'checks',results['checks_passed'],'/',results['checks_total'])


Certificate: /mnt/data/GVH_CHH_INVERSE_02B_Full_NBody_Noisy_Positions_Gate_FAST_results.json checks 16 / 16
